<a href="https://colab.research.google.com/github/KTsama07/Doc.chat/blob/main/Doc_Chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🚀 How to run this project

To run this Colab notebook successfully, you'll need to set up two API keys as **Colab Secrets**:

1.  **Google Gemini API Key** (named `GEMINI_API_KEY`)
2.  **ngrok Authentication Token** (named `NGROK_AUTH_TOKEN`)

     **Get your ngrok Authentication Token:**
        from (https://dashboard.ngrok.com/get-started/your-authtoken).

 **Ensure the **"Notebook access"** toggle is enabled for both secrets.**

If you encounter any `SecretNotFoundError` or issues with ngrok, double-check your secret names and values.


# Readme.md / The design note

### 1. Stack & Decisions
*   **LLM & Framework**: I choose a **Custom architecture** with **Gemini-3.5-Flash** (via Google GenAI SDK) to eliminate framework overhead, reduce latency, and ensure strict Pydantic-based validation of structured outputs.
*   **Data Layer:** **DuckDB** executing over a CSV backend. DuckDB provides high-performance analytical processing without a full DB server setup.
*   Execution is Constrained to **Structured Queries**. The model generates SQL, not raw Python eliminating AI genrated code execution problem.
    *   **Trade-off:** SQL is less flexible for complex multi-step data manipulation than raw Python.
*   With More Room, I would implement a **Semantic Layer** , so the LLM queries pre-calculated metrics rather than raw, messy column names.

### 2. Correctness & Trust
*   To Prevent **Fabrication** the prompt enforces a strict `is_in_scope` flag and a `Based ONLY on this data` constraint for the summary. If the SQL returns no results, the summarizer is forbidden from inventing figures.
* The UI displays the **Generated SQL** and **Raw Data** table. A skeptical officer can copy the SQL and run it manually in any standard tool to verify the number.

### 3. Government Deployment
*   **Security:** To prevent **SQL Injection** attacks, we use a `SQL_FORBIDDEN_KEYWORDS` blocklist. For a public body, we would use a **Read-Only Database User** with row-level security (RLS) to prevent data exfiltration.
*   Data is exposed to the environment by secured govt. **URL's** , system creates a copy during runtime , which gets deleted after session ends.(just for prototype), In **PRODUCTION** data would never leave the warehouse , it will be accessed through API endpoints.
*   **Audit:** The existing `audit_trail.db` would be moved to a managed service (Cloud Logging) to provide a non-repudiable record of all interactions.

### 4. Scaling
*  As data grows, DuckDB on a single CSV will hit I/O bottleneck. Manual column mapping in the prompt exceed token limits, if there are hundreds of tables.
*   **Fixes:**
    1.  Migrate to **BigQuery** or **Snowflake**.
    2.  Implementing **RAG for Metadata**: Instead of passing all column names we'll use a vector search to find relevant tables/columns based on the user's question.
    3. we can implement containerization using Docker to regulate ACCESS control over Data exposure.

### 5. Validation
*   For **Stakeholder Testing:** we'll Perform **Shadow Mode** testing—such i.e. give stakeholders the tool but have a human expert verify the SQL before they see the answer.
*   **Failure Modes:** Watch for 'Semantic Drift' (e.g., the model confusing 'accidents' with 'fatalities') and 'Join Explosions' where a poorly formed query creates massive, incorrect sums.

### 6. Honest Limitations
*   **Current Weaknesses:** The system lacks **multi-turn memory** (context from the previous question is lost). It also struggles with complex temporal logic (e.g., 'Compare the growth rate of the first half of 2019 vs 2020') which often requires CTEs that the current prompt might not consistently generate correctly.
```

### Section 1: Environment Setup
Installs necessary libraries (`duckdb`, `gradio`, `google-genai`, etc.), mounts Google Drive, and initializes the working environment.

In [ ]:
!pip install -q duckdb gradio plotly google-genai pydantic pandas
import os
import requests
def download_dataset(url, destination):
    """Downloads a file from a URL and stores it in local compute memory."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    print(f"📥 Downloading data from {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(destination, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"✅ Data saved locally to: {destination}")


print("✅ Environment ready.")

✅ Environment ready.


### Section 2: Configuration Hub
Defines global settings including file paths, LLM model selection, temperature parameters, and SQL safety blocklists.

In [ ]:
import os

class Config:
    DATA_URL = "https://drive.google.com/uc?export=download&id=1Po7yza53kB1LYPI-cHrgAWXypIGWgTOJ"

    # Use local storage for portability
    LOCAL_BASE = "/content/doc_chat_data"
    DATA_FILE = f"{LOCAL_BASE}/road_safety_clean.csv"
    AUDIT_DB = f"{LOCAL_BASE}/audit_trail.db"

    # LLM Settings - Updated to Gemini 3.1 Flash Lite
    STRUCTURED_MODEL, SUMMARY_MODEL = "gemini-3.1-flash-lite", "gemini-3.1-flash-lite"
    TEMPERATURE_STRUCTURED, TEMPERATURE_SUMMARY = 0.0, 0.3

    # Safety
    CONFIDENCE_THRESHOLD = 20
    SQL_FORBIDDEN_KEYWORDS = ["DROP", "DELETE", "INSERT", "UPDATE", "ATTACH", "COPY", "EXPORT", "ALTER", "CREATE"]

os.makedirs(Config.LOCAL_BASE, exist_ok=True)
print(f"✅ System storage initialized at: {Config.LOCAL_BASE}")
print(f"🤖 Active Models: {Config.STRUCTURED_MODEL}")

✅ System storage initialized at: /content/doc_chat_data
🤖 Active Models: gemini-3.1-flash-lite


### Section 3: Dataset Registry
Manages dataset metadata and provides a mechanism to automatically load column headers from CSV files for the active dataset.

In [ ]:
# ── Cell 3: Dataset Registry & Auto-Downloader ──────────────────────────────
import pandas as pd
import requests
from dataclasses import dataclass, field

@dataclass
class DatasetConfig:
    name:             str
    file_path:        str
    scope_description: str
    display_name:     str
    url:              str = ""
    columns:          list = field(default_factory=list)

    def _ensure_file_exists(self):
        """Download file if missing from local runtime."""
        if not os.path.exists(self.file_path):
            if not self.url:
                print(f"⚠️ Warning: {self.file_path} missing and no URL provided.")
                return False

            print(f"📥 Downloading dataset from: {self.url}...")
            try:
                r = requests.get(self.url, timeout=30)
                r.raise_for_status()
                with open(self.file_path, 'wb') as f:
                    f.write(r.content)
                print("✅ Download complete.")
            except Exception as e:
                print(f"❌ Failed to download: {e}")
                return False
        return True

    def load_columns(self):
        """Read column headers once."""
        if self._ensure_file_exists() and not self.columns:
            self.columns = pd.read_csv(self.file_path, nrows=0).columns.tolist()
        return self.columns

# ── Registry ──────────────────────────────────────────────────────────────────
DATASET_REGISTRY = {
    "road_safety": DatasetConfig(
        name="road_safety",
        file_path=Config.DATA_FILE,
        url=Config.DATA_URL,
        scope_description="road safety, traffic accidents, vehicle crashes, road fatalities, injuries, MORTH data",
        display_name="🚗 MORTH Road Safety Dataset"
    ),
}

# Active dataset initialization
ACTIVE_DATASET = DATASET_REGISTRY["road_safety"]
ACTIVE_DATASET.load_columns()

print(f"✅ Dataset Ready: {ACTIVE_DATASET.display_name}")
if ACTIVE_DATASET.columns:
    print(f"   Columns ({len(ACTIVE_DATASET.columns)}): {ACTIVE_DATASET.columns[:6]} ...")

📥 Downloading dataset from: https://drive.google.com/uc?export=download&id=1Po7yza53kB1LYPI-cHrgAWXypIGWgTOJ...
✅ Download complete.
✅ Dataset Ready: 🚗 MORTH Road Safety Dataset
   Columns (11): ['Country', 'State', 'Year', 'Number Of Railway Traffic Accident Cases Reported (UOM:Number), Scaling Factor:1', 'Number Of Road Traffic Accident Cases Reported (UOM:Number), Scaling Factor:1', 'Number Of Persons Injured In Road Traffic Accidents (UOM:Number), Scaling Factor:1'] ...


### Section 4: Pydantic Schema
Defines the structured data format (Pydantic models) used for communication between the LLM and the backend logic.

In [ ]:
# ── Cell 4: Pydantic Schema ───────────────────────────────────────────────────
from pydantic import BaseModel, Field
from typing import Literal

class QueryParameters(BaseModel):
    """Strictly typed contract between the LLM and the backend pipeline."""
    is_in_scope: bool = Field(description="True if the question is about road safety, accidents, or vehicles.")
    refusal_reason: str = Field(description="If out of scope, explain why gracefully.")
    ambiguity_analysis: str = Field(description="Critically analyze the user's wording BEFORE writing SQL. List every assumption you are making about vague terms. If you have to map a vague term to a specific column , state that explicitly here.")
    confidence_score: int = Field(description="Based strictly on the ambiguity_analysis above, rate your confidence (0-100). If you had to guess any metric mappings, this score MUST be below 20.")
    sql_query: str = Field(description="Safe DuckDB SQL SELECT statement using full table path.")
    chart_type: Literal["bar", "line", "none"] = Field(description="Best chart type for results.")
    x_axis: str = Field(description="Column name or alias for X-axis.")
    y_axis: str = Field(description="Column name or alias for Y-axis.")

print("✅ Schema defined.")

✅ Schema defined.


### Section 5: Audit Trail
Implements persistent logging using SQLite on Google Drive to track every user question, generated SQL, and system status.

In [ ]:
# ── Cell 5: Audit Trail (Persistent SQLite on Drive) ─────────────────────────
import sqlite3
from datetime import datetime, timezone

def _init_audit_db():
    with sqlite3.connect(Config.AUDIT_DB) as conn:
        conn.execute("CREATE TABLE IF NOT EXISTS audit_log (id INTEGER PRIMARY KEY AUTOINCREMENT, timestamp TEXT NOT NULL, question TEXT NOT NULL, dataset TEXT, confidence INTEGER, sql_query TEXT, status TEXT NOT NULL)")

def log_interaction(question: str, confidence: int, sql: str, status: str, dataset: str = ""):
    with sqlite3.connect(Config.AUDIT_DB) as conn:
        conn.execute("INSERT INTO audit_log (timestamp, question, dataset, confidence, sql_query, status) VALUES (?, ?, ?, ?, ?, ?)",
                     (datetime.now(timezone.utc).isoformat(), question, dataset, confidence, sql, status))

def get_audit_log(limit: int = 20) -> pd.DataFrame:
    with sqlite3.connect(Config.AUDIT_DB) as conn:
        return pd.read_sql_query(f"SELECT timestamp, question, confidence, status FROM audit_log ORDER BY id DESC LIMIT {limit}", conn)

_init_audit_db()
print(f"✅ Audit DB ready: {Config.AUDIT_DB}")

✅ Audit DB ready: /content/doc_chat_data/audit_trail.db


### Section 6: SQL Engine
Handles SQL validation (preventing unsafe commands) and executes queries against the CSV data using DuckDB.

In [ ]:
# ── Cell 6: SQL Engine (Validator + Executor) ─────────────────────────────────
import duckdb, re
class SQLValidationError(ValueError): pass

def validate_sql(sql: str, allowed_path: str) -> str:
    stripped, upper = sql.strip(), sql.strip().upper()
    if not upper.startswith("SELECT"): raise SQLValidationError(f"Only SELECT allowed. Received: '{stripped[:60]}'")
    for kw in Config.SQL_FORBIDDEN_KEYWORDS:
        if re.search(rf"\b{kw}\b", upper): raise SQLValidationError(f"Forbidden keyword: '{kw}'")
    if allowed_path not in stripped: raise SQLValidationError(f"Unauthorized table reference. Only '{allowed_path}' permitted.")
    return stripped

def execute_query(sql: str, allowed_path: str) -> pd.DataFrame:
    return duckdb.query(validate_sql(sql, allowed_path)).to_df()

print("✅ SQL Engine ready.")

✅ SQL Engine ready.


### Section 7: Prompt Builder
Constructs dynamic system instructions for the LLM based on the schema and columns of the currently active dataset.

In [ ]:
# ── Cell 7: Prompt Builder ────────────────────────────────────────────────────
def build_system_prompt(dataset: DatasetConfig) -> str:
    cols = ", ".join(f'"{c}"' for c in dataset.columns)
    return f"""You are a government data analyst.
DATASET: {dataset.display_name} | TABLE: '{dataset.file_path}' | COLUMNS: [{cols}]
SCOPE: {dataset.scope_description}

RULES:
1. Use EXACT path: '{dataset.file_path}'
2. Cast 'Year' to INT. Cast 'Number Of...' to DOUBLE for sums.
3. For comparative queries, SELECT the entity column and include in GROUP BY.
4. Use the EXACT alias (e.g. AS Total_Deaths) for chart axes.
5. If unsure/guessing → confidence_score < 20.
6. If question is unrelated to {dataset.scope_description} → is_in_scope=False + refusal_reason."""

print("✅ Prompt Builder ready.")

✅ Prompt Builder ready.


### Section 8: LLM Client
Interfaces with the Gemini API to perform structured data extraction (SQL generation) and natural language result summarization.

In [ ]:
# ── Cell 8: LLM Client ────────────────────────────────────────────────────────
import google.genai as genai
from google.colab import userdata

# Updated to use GEMINI_API_KEY
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

def call_structured_llm(question: str, dataset: DatasetConfig) -> QueryParameters:
    prompt = f"{build_system_prompt(dataset)}\n\nUser Question: \"{question}\""
    res = client.models.generate_content(
        model=Config.STRUCTURED_MODEL, contents=prompt,
        config=genai.types.GenerateContentConfig(response_mime_type="application/json", response_schema=QueryParameters, temperature=Config.TEMPERATURE_STRUCTURED)
    )
    return QueryParameters.model_validate_json(res.text)

def generate_nl_summary(question: str, df: pd.DataFrame) -> str:
    prompt = f"Based ONLY on this data: {df.to_dict()}\nAnswer: '{question}' in 2–3 clear sentences with key numbers. No external knowledge."
    res = client.models.generate_content(model=Config.SUMMARY_MODEL, contents=prompt, config=genai.types.GenerateContentConfig(temperature=Config.TEMPERATURE_SUMMARY))
    return res.text

print("✅ LLM Client ready.")

✅ LLM Client ready.


### Section 9: Chart Engine
Generates Plotly visualizations (bar/line charts) based on the specific columns and data types returned by the SQL query.

In [ ]:
# ── Cell 9: Improved Chart Engine ──────────────────────────────────────────────
import plotly.express as px
from typing import Optional

# Helper to determine grouping column (prefer 'State' or similar categorical column)
def _get_group_col(df, x, y):
    candidates = ['State', 'Country', 'Category']
    for c in candidates:
        if c in df.columns and c != x and c != y:
            return c
    return None

_RENDERERS = {
    "bar": lambda df, x, y: px.bar(
        df, x=x, y=y,
        color=_get_group_col(df, x, y),
        title=f"{y} by {x}",
        barmode="group",
        template="plotly_dark"
    ),
    "line": lambda df, x, y: px.line(
        df, x=x, y=y,
        color=_get_group_col(df, x, y),
        title=f"{y} over {x}",
        template="plotly_dark",
        markers=True
    ),
}

def render_chart(chart_type: str, df: pd.DataFrame, x: str, y: str) -> Optional[any]:
    if chart_type == "none" or df is None or df.empty or x not in df.columns or y not in df.columns:
        return None

    # Sort by X-axis (usually Year) to ensure lines are drawn in temporal order
    if x in df.columns:
        try:
            df = df.sort_values(by=x)
        except:
            pass

    return _RENDERERS.get(chart_type, lambda *args: None)(df, x, y)

print("✅ Improved Chart Engine ready (Multi-entity support enabled).")

✅ Improved Chart Engine ready (Multi-entity support enabled).


### Section 10: Orchestrator
The central logic hub that connects the LLM, SQL Engine, and UI, managing the end-to-end flow from question to answer.

In [ ]:
# ── Cell 10: Orchestrator ─────────────────────────────────────────────────────
def orchestrate(user_question: str) -> tuple:
    ds = ACTIVE_DATASET
    try:
        p = call_structured_llm(user_question, ds)
        if not p.is_in_scope:
            log_interaction(user_question, 0, "NONE", "REFUSED", ds.name)
            return p.refusal_reason, None, None, "No query.", 0
        if p.confidence_score < Config.CONFIDENCE_THRESHOLD:
            log_interaction(user_question, p.confidence_score, p.sql_query, "FLAGGED", ds.name)
            return f"⚠️ Human Intervention Needed. The system is unsure. Confidence = ({p.confidence_score}%): {p.ambiguity_analysis}", None, None, p.sql_query, p.confidence_score

        df = execute_query(p.sql_query, ds.file_path)
        ans = generate_nl_summary(user_question, df)
        fig = render_chart(p.chart_type, df, p.x_axis, p.y_axis)
        log_interaction(user_question, p.confidence_score, p.sql_query, "SUCCESS", ds.name)
        return ans, df, fig, p.sql_query, p.confidence_score
    except Exception as e:
        log_interaction(user_question, 0, "", f"ERROR: {e}", ds.name)
        return f"❌ Error: {e}", None, None, "", 0

print("✅ Orchestrator ready.")

✅ Orchestrator ready.


### Section 11: FastAPI + Tunnel Dependencies
Installs the necessary web server and tunneling libraries (`fastapi`, `uvicorn`, `pyngrok`) for the API backend.

In [ ]:
# ── Cell 11: FastAPI + Tunnel Dependencies ────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok python-multipart

print("✅ FastAPI stack ready.")

✅ FastAPI stack ready.


### Section 12: API Models
Defines the Pydantic data structures used by the FastAPI backend to validate incoming requests and ensure consistent JSON responses for the frontend, including audit logs and dataset metadata.

In [ ]:
# ── Cell 12: API Models ──────────────────────────────────────────────────────
from pydantic import BaseModel as _BM
from typing import Optional, Any

class QueryRequest(_BM): question: str

class QueryResponse(_BM):
    answer: str; sql_query: str; confidence_score: int; status: str
    table_data: list[dict]; chart_json: Optional[Any] = None

class AuditEntry(_BM): timestamp: str; question: str; confidence: Optional[int]; status: str

class DatasetInfo(_BM): name: str; columns: list[str]; scope: str

print("✅ API models defined.")

✅ API models defined.


### Section 13: FastAPI Application
The main backend application defining endpoints for health checks, queries, and serving the embedded frontend.

In [ ]:
# ── Cell 13: FastAPI Application (Optimized) ──────────────────────────────────
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
import json, plotly

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"])

@app.get("/health")
def health(): return {"status": "ok", "ds": ACTIVE_DATASET.display_name}

@app.post("/api/query", response_model=QueryResponse)
def query_api(req: QueryRequest):
    ans, df, fig, sql, conf = orchestrate(req.question)
    chart = json.loads(plotly.io.to_json(fig)) if fig else None
    table = df.fillna("").to_dict(orient="records") if df is not None else []
    status = "error" if "❌" in ans else "refused" if "No query" in sql else "success"
    return QueryResponse(answer=ans, sql_query=sql, confidence_score=conf, status=status, table_data=table, chart_json=chart)

@app.get("/api/audit")
def audit_api(limit: int = 20):
    """Endpoint to serve the audit log data to the frontend."""
    df = get_audit_log(limit)
    return df.to_dict(orient="records")

@app.get("/api/dataset/info", response_model=DatasetInfo, tags=["Meta"])
def dataset_info():
    return DatasetInfo(
        name    = ACTIVE_DATASET.display_name,
        columns = ACTIVE_DATASET.columns,
        scope   = ACTIVE_DATASET.scope_description,
    )


@app.get("/", response_class=HTMLResponse)
def home(): return FRONTEND_HTML

print("✅ API Optimized with Audit Endpoint.")

✅ API Optimized with Audit Endpoint.


### Section 14: Frontend HTML
A complete web dashboard source string, including JS logic to handle API requests and Plotly visualizations.

In [ ]:
# ── Cell 16: Frontend HTML ────────────────────────────────────────────────────
FRONTEND_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <meta name="description" content="Doc.Chat — Query government datasets with plain English.">
  <title>Doc.Chat · Data Intelligence</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:opsz,wght@9..40,300;9..40,400;9..40,500;9..40,600&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <script src="https://cdn.plot.ly/plotly-2.32.0.min.js" charset="utf-8"></script>
  <script src="https://cdn.jsdelivr.net/npm/marked@9/marked.min.js"></script>
  <style>
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

    :root {
      --paper:     #FAF8F5;
      --paper-2:   #F2EEE7;
      --paper-3:   #E8E2D8;
      --rule:      #D4CEC3;
      --ink:       #1C1814;
      --ink-2:     #4A4540;
      --ink-3:     #8C8680;
      --accent:    #C44D18;
      --accent-lt: #FEF0E8;
      --accent-dk: #9B3A10;
      --navy:      #1D3461;
      --success:   #2D6A4F;
      --warn:      #92400E;
      --danger:    #881C1C;
      --sw:        260px;
    }

    html { font-size: 15px; scroll-behavior: smooth; }
    body { font-family: 'DM Sans', system-ui, sans-serif; background: var(--paper); color: var(--ink); min-height: 100vh; }

    .top-bar { height: 3px; background: linear-gradient(90deg, var(--accent), #E8701A 50%, var(--navy)); position: fixed; top: 0; left: 0; right: 0; z-index: 100; }
    .shell { display: grid; grid-template-columns: var(--sw) 1fr; min-height: 100vh; padding-top: 3px; }

    .sidebar { position: sticky; top: 3px; height: calc(100vh - 3px); display: flex; flex-direction: column; border-right: 1px solid var(--rule); background: #fff; overflow-y: auto; padding: 1.75rem 1.4rem; }
    .brand { margin-bottom: 1.75rem; padding-bottom: 1.5rem; border-bottom: 1px solid var(--rule); }
    .brand-name { font-family: 'DM Serif Display', Georgia, serif; font-size: 1.55rem; color: var(--accent); letter-spacing: -0.02em; line-height: 1; margin-bottom: 0.2rem; }
    .brand-sub { font-size: 0.68rem; color: var(--ink-3); letter-spacing: 0.09em; text-transform: uppercase; font-weight: 500; }
    .sb-label { font-size: 0.67rem; font-weight: 600; letter-spacing: 0.1em; text-transform: uppercase; color: var(--ink-3); margin-bottom: 0.6rem; }
    .sb-sec { margin-bottom: 1.5rem; }
    .ds-card { border: 1px solid var(--rule); border-left: 3px solid var(--accent); border-radius: 4px; padding: 0.6rem 0.8rem; background: var(--accent-lt); }
    .ds-name { font-size: 0.82rem; font-weight: 600; color: var(--ink); margin-bottom: 0.2rem; }
    .ds-scope { font-size: 0.7rem; color: var(--ink-3); line-height: 1.4; }
    .recent-list { display: flex; flex-direction: column; gap: 0.25rem; }
    .r-item { font-size: 0.76rem; color: var(--ink-2); padding: 0.4rem 0.55rem; border-radius: 4px; cursor: pointer; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; border: 1px solid transparent; transition: all 0.15s; display: flex; align-items: center; gap: 0.4rem; }
    .r-item:hover { background: var(--paper-2); border-color: var(--rule); }
    .r-dot { flex-shrink: 0; width: 6px; height: 6px; border-radius: 50%; }
    .r-s .r-dot { background: var(--success); } .r-f .r-dot { background: var(--warn); }
    .r-e .r-dot { background: var(--danger); }  .r-r .r-dot { background: var(--ink-3); }
    .empty-hint { font-size: 0.74rem; color: var(--ink-3); font-style: italic; }
    .sb-foot { margin-top: auto; padding-top: 1.25rem; border-top: 1px solid var(--rule); }
    .sb-stat { font-size: 0.72rem; color: var(--ink-3); margin-bottom: 0.55rem; }
    .audit-link { font-size: 0.77rem; color: var(--accent); background: none; border: none; cursor: pointer; font-family: inherit; text-decoration: underline; text-underline-offset: 2px; padding: 0; }
    .audit-link:hover { color: var(--accent-dk); }
    .mob-hdr { display: none; }

    .main { padding: 2.5rem 2.75rem; max-width: 860px; }

    .q-eye { font-size: 0.68rem; font-weight: 600; letter-spacing: 0.1em; text-transform: uppercase; color: var(--ink-3); margin-bottom: 0.55rem; }
    .q-box { border: 1px solid var(--rule); border-radius: 8px; background: #fff; overflow: hidden; transition: border-color 0.2s, box-shadow 0.2s; margin-bottom: 1.75rem; }
    .q-box:focus-within { border-color: var(--accent); box-shadow: 0 0 0 3px rgba(196,77,24,0.08); }
    .q-input { width: 100%; border: none; outline: none; resize: none; font-family: 'DM Sans', sans-serif; font-size: 1rem; color: var(--ink); background: transparent; padding: 1rem 1.1rem 0.5rem; line-height: 1.55; }
    .q-input::placeholder { color: var(--ink-3); }
    .q-foot { display: flex; align-items: center; justify-content: space-between; padding: 0.5rem 0.6rem 0.5rem 1.1rem; border-top: 1px solid var(--paper-3); gap: 0.6rem; flex-wrap: wrap; }
    .chips-row { display: flex; gap: 0.35rem; flex-wrap: wrap; flex: 1; }
    .chip { font-size: 0.72rem; padding: 0.2rem 0.65rem; border: 1px solid var(--rule); border-radius: 100px; color: var(--ink-2); background: var(--paper-2); cursor: pointer; font-family: inherit; transition: all 0.15s; white-space: nowrap; }
    .chip:hover { border-color: var(--accent); color: var(--accent); background: var(--accent-lt); }
    .run-btn { display: inline-flex; align-items: center; gap: 0.4rem; background: var(--accent); color: #fff; border: none; border-radius: 6px; padding: 0.52rem 1.1rem; font-size: 0.84rem; font-weight: 600; cursor: pointer; font-family: inherit; transition: background 0.15s, transform 0.1s; white-space: nowrap; flex-shrink: 0; }
    .run-btn:hover { background: var(--accent-dk); } .run-btn:active { transform: scale(0.97); }
    .run-btn:disabled { background: var(--ink-3); cursor: not-allowed; transform: none; }

    .hidden { display: none !important; }
    .loading { margin-bottom: 1.5rem; }
    .prog-track { height: 2px; background: var(--paper-3); border-radius: 2px; overflow: hidden; margin-bottom: 0.7rem; }
    .prog-fill { height: 100%; background: var(--accent); width: 0%; border-radius: 2px; animation: prog 3.2s ease-in-out forwards; }
    @keyframes prog { 0%{width:0%} 30%{width:38%} 60%{width:68%} 85%{width:87%} 100%{width:92%} }
    .prog-steps { display: flex; align-items: center; gap: 0.35rem; font-size: 0.74rem; color: var(--ink-3); }
    .pstep { transition: color 0.3s; } .pstep.on { color: var(--accent); font-weight: 500; }
    .psep { color: var(--rule); }

    .results { animation: fadeUp 0.35s ease; }
    @keyframes fadeUp { from{opacity:0;transform:translateY(10px)} to{opacity:1;transform:translateY(0)} }
    .divider { height: 1px; background: var(--rule); margin: 1.5rem 0; }

    .r-meta { display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 0.75rem; margin-bottom: 1.5rem; }
    .conf-wrap { display: flex; align-items: center; gap: 0.6rem; }
    .meta-lbl { font-size: 0.68rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.09em; color: var(--ink-3); }
    .conf-track { width: 110px; height: 4px; background: var(--paper-3); border-radius: 4px; overflow: hidden; }
    .conf-fill { height: 100%; border-radius: 4px; transition: width 0.6s ease; }
    .conf-pct { font-size: 0.8rem; font-weight: 600; min-width: 2.4rem; color: var(--ink); }
    .meta-tags { display: flex; align-items: center; gap: 0.5rem; }
    .tag { font-size: 0.71rem; font-weight: 600; padding: 0.2rem 0.6rem; border-radius: 3px; border-left: 3px solid; }
    .t-ok  { background:#EDF7F3; border-color:var(--success); color:var(--success); }
    .t-no  { background:var(--paper-2); border-color:var(--ink-3); color:var(--ink-3); }
    .t-fl  { background:#FEF6E8; border-color:var(--warn); color:var(--warn); }
    .t-err { background:#FDF0F0; border-color:var(--danger); color:var(--danger); }
    .elapsed { font-size: 0.71rem; color: var(--ink-3); font-variant-numeric: tabular-nums; }

    .eye { font-size: 0.67rem; font-weight: 600; letter-spacing: 0.1em; text-transform: uppercase; color: var(--ink-3); margin-bottom: 0.7rem; }
    .answer-text { font-family: 'DM Serif Display', Georgia, serif; font-size: 1.18rem; line-height: 1.65; color: var(--ink); max-width: 680px; }
    .answer-text p { margin-bottom: 0.45rem; }
    .answer-text strong { font-family: 'DM Sans', sans-serif; color: var(--accent-dk); font-weight: 600; }

    .sql-hdr { display: flex; align-items: center; justify-content: space-between; margin-bottom: 0.55rem; }
    .sql-pre { background: var(--paper-2); border: 1px solid var(--rule); border-radius: 6px; padding: 0.9rem 1rem; font-family: 'JetBrains Mono', monospace; font-size: 0.77rem; color: #5B3B2F; white-space: pre-wrap; word-break: break-all; line-height: 1.65; overflow-x: auto; }
    .copy-btn { font-size: 0.71rem; padding: 0.18rem 0.55rem; border: 1px solid var(--rule); border-radius: 4px; background: #fff; color: var(--ink-2); cursor: pointer; font-family: inherit; transition: all 0.15s; }
    .copy-btn:hover { border-color: var(--accent); color: var(--accent); }

    #chart-area { min-height: 380px; }

    .tbl-hdr { display: flex; align-items: center; justify-content: space-between; margin-bottom: 0.55rem; }
    .tgl-btn { font-size: 0.71rem; color: var(--ink-3); background: none; border: none; cursor: pointer; font-family: inherit; text-decoration: underline; text-underline-offset: 2px; }
    .tbl-wrap { overflow-x: auto; }
    .row-pill { font-size: 0.67rem; background: var(--paper-2); border: 1px solid var(--rule); padding: 0.1rem 0.5rem; border-radius: 100px; color: var(--ink-3); margin-left: 0.4rem; }
    table { width: 100%; border-collapse: collapse; font-size: 0.8rem; }
    thead tr { border-bottom: 2px solid var(--ink); }
    th { padding: 0.5rem 0.75rem; text-align: left; font-size: 0.67rem; font-weight: 600; letter-spacing: 0.07em; text-transform: uppercase; color: var(--ink-2); }
    td { padding: 0.5rem 0.75rem; border-bottom: 1px solid var(--paper-3); color: var(--ink); }
    tbody tr:hover td { background: var(--paper-2); }
    tbody tr:last-child td { border-bottom: none; }

    .audit-overlay { position: fixed; inset: 0; z-index: 200; background: rgba(28,24,20,0.45); display: flex; align-items: center; justify-content: center; padding: 1rem; }
    .audit-modal { background: #fff; border-radius: 8px; width: 100%; max-width: 760px; max-height: 80vh; display: flex; flex-direction: column; box-shadow: 0 24px 64px rgba(0,0,0,0.18); overflow: hidden; }
    .audit-mhdr { display: flex; align-items: center; justify-content: space-between; padding: 1rem 1.25rem; border-bottom: 2px solid var(--ink); }
    .audit-mhdr h2 { font-family: 'DM Serif Display', serif; font-size: 1.1rem; }
    .audit-close { background: none; border: none; font-size: 1rem; cursor: pointer; color: var(--ink-3); }
    .audit-close:hover { color: var(--ink); }
    .audit-mbody { padding: 1rem 1.25rem; overflow-y: auto; }
    .refresh-btn { font-size: 0.77rem; padding: 0.32rem 0.8rem; border: 1px solid var(--rule); border-radius: 4px; background: var(--paper-2); color: var(--ink-2); cursor: pointer; font-family: inherit; margin-bottom: 1rem; transition: all 0.15s; }
    .refresh-btn:hover { border-color: var(--accent); color: var(--accent); }

    @media (max-width: 768px) {
      .shell { grid-template-columns: 1fr; }
      .sidebar { position: static; height: auto; border-right: none; border-bottom: 1px solid var(--rule); padding: 1rem 1.1rem; display: none; }
      .sidebar.open { display: flex; }
      .mob-hdr { display: flex; align-items: center; justify-content: space-between; padding: 0.8rem 1.1rem; background: #fff; border-bottom: 1px solid var(--rule); position: sticky; top: 3px; z-index: 10; }
      .mob-brand { font-family: 'DM Serif Display', serif; font-size: 1.1rem; color: var(--accent); }
      .mob-menu { background: none; border: none; font-size: 1.1rem; cursor: pointer; color: var(--ink-2); }
      .main { padding: 1.25rem 1.1rem; }
    }
    @media (min-width: 769px) { .mob-hdr { display: none; } }
  </style>
</head>
<body>

<div class="top-bar"></div>

<div class="mob-hdr">
  <span class="mob-brand">Doc.Chat</span>
  <button class="mob-menu" onclick="document.getElementById('sb').classList.toggle('open')" aria-label="Menu">&#9776;</button>
</div>

<div class="shell">
  <aside class="sidebar" id="sb">
    <div class="brand">
      <div class="brand-name">Doc.Chat</div>
      <div class="brand-sub">Government Data Intelligence</div>
    </div>
    <div class="sb-sec">
      <div class="sb-label">Active Dataset</div>
      <div class="ds-card">
        <div class="ds-name" id="ds-name">Loading&hellip;</div>
        <div class="ds-scope" id="ds-scope">&mdash;</div>
      </div>
    </div>
    <div class="sb-sec" style="flex:1;overflow:hidden">
      <div class="sb-label">Query History</div>
      <div class="recent-list" id="r-list">
        <span class="empty-hint">No queries yet</span>
      </div>
    </div>
    <div class="sb-foot">
      <div class="sb-stat" id="q-count">0 queries this session</div>
      <button class="audit-link" onclick="openAudit()">View full audit log &rarr;</button>
    </div>
  </aside>

  <main class="main">
    <div class="q-eye">Ask a question</div>
    <div class="q-box">
      <textarea id="q" class="q-input" rows="2"
        placeholder="Which state had the most road fatalities in 2020?"></textarea>
      <div class="q-foot">
        <div class="chips-row">
          <button class="chip" onclick="setQ(this)">Fatalities by state, 2019</button>
          <button class="chip" onclick="setQ(this)">Accident trend 2015&ndash;2020</button>
          <button class="chip" onclick="setQ(this)">Maharashtra vs Tamil Nadu</button>
          <button class="chip" onclick="setQ(this)">Top 5 states, 2018</button>
        </div>
        <button class="run-btn" id="run-btn" onclick="analyze()">
          Run Query
          <svg width="13" height="13" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round">
            <line x1="5" y1="12" x2="19" y2="12"/><polyline points="12 5 19 12 12 19"/>
          </svg>
        </button>
      </div>
    </div>

    <div id="loading" class="loading hidden" aria-live="polite">
      <div class="prog-track"><div class="prog-fill" id="pfill"></div></div>
      <div class="prog-steps">
        <span class="pstep" id="ps1">Analyzing intent</span>
        <span class="psep">&middot;</span>
        <span class="pstep" id="ps2">Generating SQL</span>
        <span class="psep">&middot;</span>
        <span class="pstep" id="ps3">Querying dataset</span>
        <span class="psep">&middot;</span>
        <span class="pstep" id="ps4">Assembling answer</span>
      </div>
    </div>

    <div id="results" class="results hidden">
      <div class="r-meta">
        <div class="conf-wrap">
          <span class="meta-lbl">Confidence</span>
          <div class="conf-track"><div class="conf-fill" id="cfill"></div></div>
          <span class="conf-pct" id="cpct">&mdash;</span>
        </div>
        <div class="meta-tags">
          <span class="tag" id="stag">&mdash;</span>
          <span class="elapsed" id="el">&mdash;</span>
        </div>
      </div>
      <div class="divider"></div>
      <div class="eye">Answer</div>
      <div class="answer-text" id="ans"></div>
      <div class="divider"></div>
      <div class="sql-hdr">
        <div class="eye" style="margin:0">Generated SQL</div>
        <button class="copy-btn" id="copy-btn" onclick="copySQL()">Copy</button>
      </div>
      <pre class="sql-pre" id="sql"></pre>
      <div id="chart-sec" style="display:none">
        <div class="divider"></div>
        <div class="eye">Visualization</div>
        <div id="chart-area"></div>
      </div>
      <div id="tbl-sec" style="display:none">
        <div class="divider"></div>
        <div class="tbl-hdr">
          <div class="eye" style="margin:0">Data<span class="row-pill" id="rpill"></span></div>
          <button class="tgl-btn" onclick="tglTable()">Show / Hide</button>
        </div>
        <div id="tbl-wrap" class="tbl-wrap"><table id="dtable"></table></div>
      </div>
    </div>
  </main>
</div>

<div id="audit-ov" class="audit-overlay hidden" onclick="if(event.target===this)closeAudit()">
  <div class="audit-modal">
    <div class="audit-mhdr">
      <h2>Audit Trail</h2>
      <button class="audit-close" onclick="closeAudit()">&#10005;</button>
    </div>
    <div class="audit-mbody">
      <button class="refresh-btn" onclick="loadAudit()">&#8635; Refresh</button>
      <div class="tbl-wrap"><table id="atable"></table></div>
    </div>
  </div>
</div>

<script>
  const API = window.location.origin;
  let sessN = 0;
  const hist = [];

  (async () => {
    try {
      const d = await fetch(`${API}/api/dataset/info`).then(r => r.json());
      document.getElementById('ds-name').textContent  = d.name;
      document.getElementById('ds-scope').textContent = d.scope.slice(0, 65) + '\\u2026';
    } catch (_) {}
  })();

  function setQ(btn) { document.getElementById('q').value = btn.textContent; analyze(); }

  async function copySQL() {
    await navigator.clipboard.writeText(document.getElementById('sql').textContent);
    const b = document.getElementById('copy-btn');
    b.textContent = 'Copied'; setTimeout(() => b.textContent = 'Copy', 2000);
  }

  let _t;
  function startLoad() {
    document.getElementById('loading').classList.remove('hidden');
    document.getElementById('results').classList.add('hidden');
    document.getElementById('run-btn').disabled = true;
    const old = document.getElementById('pfill');
    const nw  = old.cloneNode(); nw.id = 'pfill'; old.replaceWith(nw);
    ['ps1','ps2','ps3','ps4'].forEach(id => document.getElementById(id).classList.remove('on'));
    let i = 0;
    _t = setInterval(() => { if (i < 4) document.getElementById(`ps${++i}`).classList.add('on'); }, 750);
  }
  function stopLoad() {
    clearInterval(_t);
    document.getElementById('loading').classList.add('hidden');
    document.getElementById('run-btn').disabled = false;
  }

  async function analyze() {
    const q = document.getElementById('q').value.trim();
    if (!q) return;
    startLoad();
    const t0 = Date.now();
    try {
      const data = await fetch(`${API}/api/query`, {
        method: 'POST', headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({question: q}),
      }).then(r => { if (!r.ok) throw new Error(`HTTP ${r.status}`); return r.json(); });
      stopLoad();
      render(data, Date.now() - t0);
      addHistory(q, data.status);
    } catch (err) { stopLoad(); renderErr(err.message); }
  }

  const STATUS = {
    success: ['\\u2713 Success',       't-ok'],
    refused: ['\\u2014 Out of Scope', 't-no'],
    flagged: ['\\u26a0 Flagged',       't-fl'],
    error:   ['\\u2717 Error',         't-err'],
  };

  function render(d, ms) {
    document.getElementById('results').classList.remove('hidden');
    const sc = d.confidence_score ?? 0;
    const cf = document.getElementById('cfill');
    cf.style.width      = `${sc}%`;
    cf.style.background = sc >= 70 ? '#2D6A4F' : sc >= 35 ? '#92400E' : '#881C1C';
    document.getElementById('cpct').textContent = `${sc}%`;
    const [lbl, cls] = STATUS[d.status] ?? ['Unknown', ''];
    const st = document.getElementById('stag');
    st.textContent = lbl; st.className = `tag ${cls}`;
    document.getElementById('el').textContent = `${(ms/1000).toFixed(1)}s`;
    document.getElementById('ans').innerHTML = marked.parse(String(d.answer ?? ''));
    document.getElementById('sql').textContent = d.sql_query ?? '';
    const cs = document.getElementById('chart-sec');
    if (d.chart_json?.data) {
      cs.style.display = 'block';
      Plotly.react('chart-area', d.chart_json.data, d.chart_json.layout ?? {}, {responsive:true});
    } else { cs.style.display = 'none'; }
    const ts = document.getElementById('tbl-sec');
    if (d.table_data?.length) {
      ts.style.display = 'block';
      document.getElementById('rpill').textContent = `${d.table_data.length} rows`;
      buildTable('dtable', d.table_data);
    } else { ts.style.display = 'none'; }
  }

  function renderErr(msg) {
    document.getElementById('results').classList.remove('hidden');
    document.getElementById('ans').textContent = `Error: ${msg}`;
    const st = document.getElementById('stag'); st.textContent = '\\u2717 Error'; st.className = 'tag t-err';
  }

  function buildTable(id, rows) {
    const cols  = Object.keys(rows[0]);
    const thead = `<thead><tr>${cols.map(c=>`<th>${c}</th>`).join('')}</tr></thead>`;
    const tbody = `<tbody>${rows.map(r=>`<tr>${cols.map(c=>`<td>${r[c]??''}</td>`).join('')}</tr>`).join('')}</tbody>`;
    document.getElementById(id).innerHTML = thead + tbody;
  }

  let tblVis = true;
  function tglTable() { tblVis=!tblVis; document.getElementById('tbl-wrap').style.display=tblVis?'':'none'; }

  function addHistory(q, status) {
    sessN++;
    document.getElementById('q-count').textContent = `${sessN} quer${sessN===1?'y':'ies'} this session`;
    hist.unshift({q, status});
    if (hist.length > 7) hist.pop();
    const cls = {success:'r-s', refused:'r-r', flagged:'r-f', error:'r-e'};
    document.getElementById('r-list').innerHTML = hist.map(({q,status}) =>
      `<div class="r-item ${cls[status]??''}" onclick="document.getElementById('q').value=this.dataset.q;analyze()" data-q="${q.replace(/"/g,'&quot;')}">
         <span class="r-dot"></span><span style="overflow:hidden;text-overflow:ellipsis">${q}</span>
       </div>`
    ).join('');
  }

  function openAudit()  { document.getElementById('audit-ov').classList.remove('hidden'); loadAudit(); }
  function closeAudit() { document.getElementById('audit-ov').classList.add('hidden'); }

  async function loadAudit() {
    try {
      const rows = await fetch(`${API}/api/audit?limit=20`).then(r => r.json());
      if (!rows.length) { document.getElementById('atable').innerHTML='<tr><td>No entries yet.</td></tr>'; return; }
      buildTable('atable', rows);
    } catch (e) { document.getElementById('atable').innerHTML=`<tr><td>Failed: ${e.message}</td></tr>`; }
  }

  document.getElementById('q').addEventListener('keydown', e => {
    if (e.key === 'Enter' && (e.metaKey || e.ctrlKey)) analyze();
  });
  document.addEventListener('keydown', e => { if (e.key==='Escape') closeAudit(); });
</script>
</body>
</html>"""

print(f"✅ Frontend HTML defined ({len(FRONTEND_HTML):,} bytes).")

✅ Frontend HTML defined (22,638 bytes).


### Section 15: Launch & Tunneling
Initializes the ngrok tunnel and starts the background web server to make the application publicly accessible.

In [ ]:
# ── Cell 17: Launch ───────────────────────────────────────────────────────────
import threading, time, uvicorn, os
from pyngrok import ngrok, conf
from google.colab import userdata

# 1. Configuration & Port Cleanup
_PORT = 8000
os.system(f"fuser -k {_PORT}/tcp") # Kill existing process on port
conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN')

# 2. Check if App exists and Start Uvicorn
try:
    # Ensure 'app' from cell 'uyIR3d_K8JTY' is available
    app_ref = globals().get('app')
    if app_ref is None:
        raise NameError("app variable not found in global namespace")

    threading.Thread(
        target = uvicorn.run,
        kwargs = dict(app=app_ref, host="0.0.0.0", port=_PORT, log_level="warning"),
        daemon = True,
    ).start()

    # 3. Establish Tunnel
    time.sleep(2)
    [ngrok.disconnect(t.public_url) for t in ngrok.get_tunnels()]
    PUBLIC_URL = ngrok.connect(_PORT).public_url

    print(f"""
{'─'*52}
💬 Doc.Chat is live!
{'─'*52}
  🌐  Frontend  →  {PUBLIC_URL}
  📋  API docs  →  {PUBLIC_URL}/docs
  🔍  Health    →  {PUBLIC_URL}/health
{'─'*52}
⚠️  Re-run this cell ONLY if you need a fresh URL.
""")
except NameError:
    print("❌ Error: 'app' is not defined. PLEASE RUN CELL 13 (FastAPI Application) FIRST.")
except Exception as e:
    print(f"❌ Launch Error: {e}")


────────────────────────────────────────────────────
💬 Doc.Chat is live!
────────────────────────────────────────────────────
  🌐  Frontend  →  https://camping-marry-bakery.ngrok-free.dev
  📋  API docs  →  https://camping-marry-bakery.ngrok-free.dev/docs
  🔍  Health    →  https://camping-marry-bakery.ngrok-free.dev/health
────────────────────────────────────────────────────
⚠️  Re-run this cell ONLY if you need a fresh URL.

